# Homework 1

CESG 561 - Autumn 2023

Author: Pedro Arduino

April 13, 2024

A site adjacent to San  Francisco Bay is underlain by San Francisco Bay Mud.  The site is to be readied for development by placement of 5 ft of fill material, and the ultimate settlement of the fill is of interest.  The site conditions, shown below, indicate the presence of a crust of desiccated Bay Mud with thickness, h1, which is not expected to consolidate noticeably.  The clay is underlain by a dense gravel, which will also not consolidate.  

<img src="./settlementProblem.png" width="400" alt="Settlement Problem" />

A subsurface investigation revealed the properties shown in the table below.  Many of the properties are uncertain, either due to scatter in the data from which they were developed, or due to the judgment of the engineers who were involved in acquisition of the data.  Nevertheless, the  uncertainty associated with each property is indicated in the table; you can assume that each is normally distributed.  Recall that you can compute unit weights from void ratios. For evaluating settlements subdivide the compressible layer into sublayers (10-20)
Note: COV (coefficient of variation) = $\sigma / \mu * 100$, $\sigma$ = standard deviation, $\mu$ = mean value.

| Parameter           | Mean value | Coefficient of variation (%) |
| :--------           | :--------: | :--------------------------: |
|     $h_1$           |     3 ft   |             5                |
|     $h_2$           |    25 ft   |             5                |
|     $C_c$           |    0.75    |            20                |
|     $C_r$           |    0.05    |            20                |
| $\sigma_p-\sigma_i$ |  200 psf   |            50                |
|     $e_o$           |    1.54    |             7                |
|   $\gamma_{fill}$   |   130 pcf  |             7                |
|   $h_{fill}$        |     5 ft   |             2                |










In [1]:
ls

sample_data/  SettlementField.png


# Settlement calculation

Import required modules:

In [ ]:
# @title Default title text
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys

from IPython.display import display

### Define settlement function

In [ ]:
# @title Default title text

def settlement(h1, h2, Cc, Cr, e0, precon_press, gammafill, hfill, n=10, alpha=0.5,retdata=False):

    load = gammafill * hfill
    layerh = (h2/n)

    z =  np.arange(n) + alpha
    z *= layerh       # multiply by layer height

    gammasoil = (2.67 + e0)*62.4/(1. + e0)

    sigi =  z*(gammasoil - 62.4)
    sigi += h1*gammasoil
    sigp =  sigi + precon_press
    sigf =  sigi + load

    hoc = Cr/(1+e0) * layerh * np.log10(sigp/sigi)
    hnc = Cc/(1+e0) * layerh * np.log10(sigf/sigp)

    ht = hoc + hnc

    if retdata:    # these are all pointers to lists, NOT copies
        data = {}
        data['Layer No.']    = np.linspace(1, n, n)
        data['Depth (ft)']   = z
        data["Initial Effective Stress, \u03C3'vi (psf)"] = sigi
        data["Preconsolidation Stress, \u03C3' (psf)"]    = sigp
        data["Final Effective Stress, \u03C3'vf (psf)"]   = sigf
        data["OC Settlement of Layer, hoc (ft)"] = hoc
        data["NC Settlement, hnc (ft)"]          = hnc
        data["Total Layer Settlement, ht (ft)"]  = ht

        return (np.sum(ht), data)
    else:
        return np.sum(ht)

### Perform settlement evaluation:

In [ ]:
h1 = 3 #ft
h2 = 25 #ft
Cc = 0.75
Cr = 0.05
e0 = 1.54
precon_press = 200 #psf greater than sigvi
gammafill = 130 #pcf
hfill = 5 #ft
n = 100
alpha = 0.7

ans, data = settlement(h1, h2, Cc, Cr, e0, precon_press, gammafill, hfill, n, alpha, True)
print("The total consolidation settlement is equal to {:.2f} feet".format(ans))

display(pd.DataFrame(data))

## Tornado Diagram of Uncertainty

Perform the calculations using a dictionary and the previously-defined Consolidation class.

### Data

In [ ]:
# Data

mean_value = pd.Series({'h1':3, 'h2':25, 'Cc':0.75, 'Cr':0.05, 'e0':1.54,
                        'precon_press':200, 'gammafill':130, 'hfill':5})

COV = pd.Series({'h1':5, 'h2':5, 'Cc':20, 'Cr':7, 'e0':7,
                        'precon_press':10, 'gammafill':7, 'hfill':2})



### Evaluate upper and lower bound VARIABLES

In [ ]:
# @title Default title text
# Evaluate upper and lower bound variables

ub_prop = []
lb_prop = []
i = 0
while i < len(mean_value):
    ub_prop_i = mean_value.iloc[i]*(1+COV.iloc[i]/100)
    lb_prop_i = mean_value.iloc[i]*(1-COV.iloc[i]/100)
    ub_prop.append(ub_prop_i)
    lb_prop.append(lb_prop_i)
    i = i + 1

data = pd.DataFrame({'Mean Value':mean_value,
                     'Coefficient of Variation':COV,
                     'Lower Bound Value':lb_prop,
                     'Upper Bound Value':ub_prop})
data

### Evaluate upper and lower bound SETTLEMENTS

In [ ]:
# @title Default title text
# Evaluate upper and lower bound SETTLEMENTS
# ==========================================
x = 0
n = 10
alpha = 0.5

mean_consol  = []
lb_consol    = []
ub_consol    = []
swing_consol = []

variables = data['Mean Value']
COV       = data['Coefficient of Variation']


while x < len(variables):
    mean_consolx = settlement(*variables,n,alpha)

    lb = variables.copy()
    lb.iloc[x] = lb_prop[x]
    lb_consolx = settlement(*lb,n,alpha)

    ub = variables.copy()
    ub.iloc[x] = ub_prop[x]
    ub_consolx = settlement(*ub,n,alpha)

    swing = abs(ub_consolx - lb_consolx)

    mean_consol.append(mean_consolx)
    lb_consol.append(lb_consolx)
    ub_consol.append(ub_consolx)
    swing_consol.append(swing)

    x = x + 1

res = pd.DataFrame({'Mean Value':mean_value,
                    'Coefficient of Variation':COV,
                    'Lower Bound Value':lb_prop,
                    'Upper Bound Value':ub_prop,
                    'Lower Bound Consolidation (ft)':lb_consol,
                    'Mean Consolidation (ft)':mean_consol,
                    'Upper Bound Consolidation (ft)':ub_consol,
                    'swing':swing_consol},)

res = res.sort_values("swing", ascending=True)
res = res.reset_index()

res

### Produce the Tornado diagram.

In [ ]:
# @title Default title text
fig, ax = plt.subplots(figsize=(10,7))

# pos_bars = [(low, base - low), neg_bars = (base, up - base)],
pos_bars = ax.barh(y=res.index, width=(res['Mean Consolidation (ft)']-res['Lower Bound Consolidation (ft)']), left=res['Lower Bound Consolidation (ft)'], color='pink',edgecolor='black', linewidth=1, label='Lower')
neg_bars = ax.barh(y=res.index, width=(res['Upper Bound Consolidation (ft)']-res['Mean Consolidation (ft)']), left=res['Mean Consolidation (ft)'], color='blue', edgecolor='black', linewidth=1, label='Upper')

# Draw a vertical line down the middle
plt.axvline(res['Mean Consolidation (ft)'][0], color='black')

# Position the x-axis on the top, hide all the other spines (=axis lines)
axes = plt.gca()  # (gca = get current axes)
axes.spines['left'].set_visible(True)
axes.spines['right'].set_visible(True)
axes.spines['bottom'].set_visible(True)
axes.xaxis.set_ticks_position('top')
axes.xaxis.set_label_position('top')

properties = res['index'].values.tolist()
ys = res.index.values.tolist()
plt.yticks(ys, properties)
#plt.ylim(-1, len(properties))

plt.xlim(res['Mean Consolidation (ft)'][0] - 0.55* res['swing'][7], res['Mean Consolidation (ft)'][0] + 0.55*res['swing'][7])
plt.ylim(-1, len(properties))

plt.xlabel("Settlement [ft]")
plt.ylabel("Property")

### Produce settlement field plot

In [ ]:
# Create 3D plot of Settlement as a function of relevant variables Cc and precon_press

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Enable interactive plots (choose one)
# %matplotlib notebook
# %matplotlib widget

    # Crete solution space
    #=======================================

i = 0
n = 10
alpha = 0.5
variables = data['Mean Value']

ppress_val = []
Cc_val = []

Cc_1, Cc_2, interval = 0.45, 1.05, 0.1
Cc_val = np.arange(Cc_1, Cc_2+1, interval)

ppress_1, ppress_2, interval = 50, 400, 1
ppress_val = np.arange(ppress_1, ppress_2+1, interval)

i = 0
j = 0

num_rows = len(Cc_val)
num_cols = len(ppress_val)
settl = np.zeros((num_rows, num_cols))

CC, PP = np.meshgrid(Cc_val, ppress_val, indexing='ij')

for i in range(num_rows):
    var = variables.copy()
    var.loc['Cc'] = Cc_val[i]
    for j in range(num_cols):
        var.loc['precon_press'] = ppress_val[j]
        sI = settlement(*var,n,alpha)
        settl[i,j] = sI

In [ ]:
# @title Default title text
from mpl_toolkits.mplot3d import Axes3D

# Enable interactive plots (choose one)
# %matplotlib notebook
# %matplotlib widget

# Create a 3D plot
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

z = np.array(settl)
# Create an array for white color (RGBA)
facecolors = np.ones((z.shape[0], z.shape[1], 4))
facecolors[..., 3] = 0  # Set alpha channel to 1 (opaque)

surf = ax.plot_surface(CC, PP, z, facecolors=facecolors, edgecolor='black', linewidth=0.3, antialiased=True)

ax.set_xlabel('Cc')
ax.set_ylabel('Precon Press')
ax.set_zlabel('Settlement')
ax.set_title('Settlemnt field for varying Cc and Precon Press')

# Add contour lines
levels = [0.88, 1.316]  # Specify the levels you want to extract
contour_lines = ax.contour(CC, PP, z, levels=levels, colors='red', linewidths=1.5, zorder=5)

# Add points to plot
# x_point0 = 0.75
# y_point0 = 200
# z_point0 = 1.326

x_point1 = 0.522
y_point1 = 200
z_point1 = 0.88

x_point2 = 0.660
y_point2 = 278.0
z_point2 = 0.88

# ax.scatter(x_point0, y_point0, z_point0, color='black', s=30, zorder=100, label='propagation point')
ax.scatter(x_point1, y_point1, z_point1, color='black', s=30, zorder=100, label='propagation point')
ax.scatter(x_point2, y_point2, z_point2, color='black', s=30, zorder=100, label='propagation point')

# Create custom legend handles
handles = [plt.Line2D([0], [0], color=contour_lines.cmap(contour_lines.norm(level)), lw=2) for level in contour_lines.levels]
labels = [f's = {level:.2f} m' for level in contour_lines.levels]
ax.legend(handles, labels, title="Settlement", loc='upper right')

ax.view_init(elev=-163, azim=172)  # Change the elevation and azimuth angles

plt.savefig('SettlementField.png')
plt.show()